# Cellpose segmentation benchmark on Xenium data

This notebook segments Xenium morphology images with **Cellpose** and evaluates the result against a reference segmentation with the shared metric implementation in `test.py`.

It supports two kinds of reference annotation:

1. `xenium_roi_crop/`: a manually annotated label mask (`manual_labels.npy`)
2. Other Xenium datasets: a label mask rasterized from the official `cell_boundaries.parquet`

**Workflow.** Read the morphology image, run Cellpose (prediction), build the reference mask, align the shapes, and compute the metrics with `test.py`.

### Note on the order of the arguments

In `test.py`, `evaluate_single_mask_against_cellpose(cellpose_gt_mask, pred_mask)` treats its first argument as the *reference*. In this benchmark, **Cellpose is the method being tested, not the reference**, so the reference mask (manual or official) is passed as the first argument and the Cellpose prediction as the second. As a result:

- recall = fraction of reference cells detected by Cellpose
- FP / over-segmentation = extra cells produced by Cellpose
- FN / under-segmentation = reference cells missed by Cellpose

The metrics therefore point in the correct direction for evaluating Cellpose. Misleading field names in the output of `test.py` are renamed below.

**Dependencies.** numpy, pandas, scikit-image, scipy, tifffile, pyarrow, and cellpose (version 4 or later, Cellpose-SAM). The metric implementation `test.py` must be available (see the next cell).

## 1. Load the metric implementation (`test.py`)

Set `TEST_PY_PATH` to the absolute path of `test.py`. If the path is empty or does not exist, the notebook falls back to a `test.py` in the current working directory or on `PYTHONPATH`.

In [ ]:
import os
import re
import sys
import json
import glob
import warnings
import importlib.util

import numpy as np
import pandas as pd
import tifffile
from skimage.draw import polygon as sk_polygon

TEST_PY_PATH = "yourpath/eval/test.py"   # change to the absolute path of your test.py


def _load_metrics():
    if TEST_PY_PATH and os.path.exists(TEST_PY_PATH):
        spec = importlib.util.spec_from_file_location("xenium_metrics", TEST_PY_PATH)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)   # the __main__ block of test.py is not triggered
        return mod
    here = os.getcwd()                 # notebooks have no __file__, so use the working directory
    if here not in sys.path:
        sys.path.insert(0, here)
    try:
        import test as mod
        return mod
    except ImportError as e:
        raise ImportError(
            "test.py not found. Set TEST_PY_PATH to its absolute path, "
            "or place test.py in the working directory of this notebook."
        ) from e


metrics = _load_metrics()   # metric implementation from test.py
print("Loaded metrics from:", getattr(metrics, "__file__", "unknown"))

## 2. Configuration

Edit the settings below. Each entry in `DATASETS` describes one dataset to segment and evaluate. `crop=(y0, y1, x0, x1)` selects a region in full-resolution pixels; alternatively, `crop_size` selects a centred square of that size.

In [ ]:
XENIUM_DEFAULT_PIXEL_SIZE = 0.2125   # um per pixel, used when the experiment file cannot be read

# Global Cellpose settings (Cellpose-SAM / version 4.x)
CELLPOSE = dict(
    use_gpu=True,
    diameter=None,            # Cellpose-SAM is scale-insensitive, so None is fine; classic models need a value (about 60 for cells, 30 for nuclei)
    flow_threshold=0.4,
    cellprob_threshold=0.0,
    batch_size=32,
    normalize=True,           # percentile normalization, important for high-dynamic-range uint16 Xenium images
    # For classic Cellpose (3.x) instead of Cellpose-SAM, set use_classic=True and choose model_type and channels
    use_classic=False,
    classic_model_type="cyto3",
    classic_channels=[2, 1],  # [cytoplasm, nucleus], where 1 = red, 2 = green, 3 = blue (see the classic Cellpose docs)
)

# Channel convention: in Xenium morphology_focus, channel 0 = DAPI (nucleus) and
# channel 1 = membrane or boundary stain (ATP1A1/CD45/E-cadherin)
NUCLEUS_IDX = 0
MEMBRANE_IDX = 1

DATASETS = [
    dict(
        name="xenium_mouse_brain",
        dataset_dir="yourpath/dataset/xenium_mouse_brain/outs",   # note: the path points to outs/
        output_dir="yourpath/cellpose/xenium_mouse_brain_output",
        gt_type="xenium_boundaries",
        boundaries="cell_boundaries.parquet",
        experiment="experiment.xenium",
        pyramid_level=0,
        use_membrane=True,
        crop=(12077, 22077, 6956, 16956),
    ),
]
OUTPUT_DIR = "yourpath/cellpose/xenium_out"
BOUNDARY_MODE = "thick"
IOU_THRESHOLD = 0.5

## 3. Read the morphology images

These helpers read a chosen pyramid level of an OME-TIFF, split multi-channel arrays into channels, discover the morphology files of a Xenium bundle, and assemble the Cellpose input.

In [ ]:
def _read_level(path, pyramid_level):
    """Read one pyramid level of an OME-TIFF and return (array, downsample_factor).

    factor = full-resolution height / height of this level, computed from the real
    shapes rather than assuming powers of two.
    """
    with tifffile.TiffFile(path) as tf:
        series = tf.series[0]
        try:
            levels = list(series.levels)
        except Exception:
            levels = [series]
        L = int(np.clip(pyramid_level, 0, len(levels) - 1))
        if L != pyramid_level:
            warnings.warn(f"{os.path.basename(path)}: only {len(levels)} pyramid levels, "
                          f"pyramid_level={pyramid_level} was clipped to {L}")
        arr = np.asarray(levels[L].asarray())
        base_h = levels[0].shape[-2]
        cur_h = levels[L].shape[-2]
        factor = float(base_h) / float(cur_h)
    return arr, factor


def _split_channels(arr):
    """Split an array into a list of 2D channels. 2D -> one channel; 3D -> the small axis (<= 5) is the channel axis."""
    if arr.ndim == 2:
        return [arr]
    if arr.ndim == 3:
        if arr.shape[0] <= 5 and arr.shape[0] <= arr.shape[-1]:
            return [arr[i] for i in range(arr.shape[0])]        # (C, H, W)
        if arr.shape[-1] <= 5:
            return [arr[..., i] for i in range(arr.shape[-1])]  # (H, W, C)
        return [arr[arr.shape[0] // 2]]                          # probably a z-stack, take the middle plane
    # Higher dimensions: squeeze, then project the leading axes
    arr = np.squeeze(arr)
    return _split_channels(arr) if arr.ndim <= 3 else [np.max(arr, axis=tuple(range(arr.ndim - 2)))]


def load_morphology_channels(dataset_dir, pyramid_level=0):
    """Discover and read the morphology channels.

    The morphology_focus/ directory is preferred, with file names such as
        ch0000_dapi.ome.tif ...            (breast naming)
        morphology_focus_0000.ome.tif ...  (ROI naming)
    A multi-channel file is expanded into its channels. If the directory is missing,
    the notebook falls back to morphology.ome.tif.

    Returns dict(channels=[2D arrays], names=[...], factor=float).
    """
    focus_dir = os.path.join(dataset_dir, "morphology_focus")
    channels, names = [], []

    if os.path.isdir(focus_dir):
        files = sorted(glob.glob(os.path.join(focus_dir, "*.ome.tif")) +
                       glob.glob(os.path.join(focus_dir, "*.ome.tiff")))

        def _idx(fp):
            b = os.path.basename(fp)
            m = re.search(r"ch0*(\d+)", b) or re.search(r"morphology_focus_0*(\d+)", b)
            return int(m.group(1)) if m else 0
        files = sorted(files, key=_idx)

        factor = 1.0
        for fp in files:
            arr, factor = _read_level(fp, pyramid_level)
            for c in _split_channels(arr):
                channels.append(c)
                names.append(os.path.splitext(os.path.basename(fp))[0])
        if channels:
            return dict(channels=channels, names=names, factor=factor)

    # Fallback: morphology.ome.tif (usually DAPI, possibly a z-stack)
    for cand in ("morphology.ome.tif", "morphology_focus.ome.tif", "morphology_mip.ome.tif"):
        fp = os.path.join(dataset_dir, cand)
        if os.path.exists(fp):
            arr, factor = _read_level(fp, pyramid_level)
            chs = _split_channels(arr)
            if len(chs) > 1:   # usually a z-stack: use the maximum projection as a single nuclear channel
                chs = [np.max(np.stack(chs, 0), 0)]
            warnings.warn(f"{dataset_dir}: morphology_focus/ not found, falling back to {cand} (nucleus only)")
            return dict(channels=chs, names=[cand], factor=factor)

    raise FileNotFoundError(f"{dataset_dir}: no morphology OME-TIFF found")


def assemble_cellpose_input(channels, use_membrane=True,
                            nucleus_idx=NUCLEUS_IDX, membrane_idx=MEMBRANE_IDX):
    """Assemble the channels into the Cellpose input.

    Returns (img, mode):
      - with a membrane channel and use_membrane -> (H, W, 2) float32 [nucleus, membrane], whole-cell segmentation
      - otherwise -> (H, W) float32 grayscale (nucleus), nuclear segmentation only
    Cellpose-SAM normalizes internally, so only the data type is changed here.
    """
    n = len(channels)
    nuc = channels[nucleus_idx] if nucleus_idx < n else channels[0]
    nuc = nuc.astype(np.float32)

    if use_membrane and membrane_idx < n:
        mem = channels[membrane_idx].astype(np.float32)
        img = np.stack([nuc, mem], axis=-1)   # (H, W, 2)
        return img, "nucleus+membrane"
    return nuc, "nucleus_only"

## 4. Run Cellpose

By default, Cellpose-SAM (version 4.x) is used. Set `use_classic=True` in the configuration to use classic Cellpose (version 3.x) instead.

In [ ]:
def run_cellpose(img, cfg=CELLPOSE):
    """Run Cellpose and return an int32 label mask."""
    from cellpose import models   # imported here so that the other cells work without cellpose installed

    common = dict(
        flow_threshold=cfg["flow_threshold"],
        cellprob_threshold=cfg["cellprob_threshold"],
        normalize=cfg["normalize"],
        diameter=cfg["diameter"],
    )

    if cfg.get("use_classic", False):
        # Classic Cellpose 3.x needs model_type and channels
        model = models.Cellpose(gpu=cfg["use_gpu"], model_type=cfg["classic_model_type"])
        # Classic models expect [cytoplasm, nucleus], so (H, W, 2) = [nucleus, membrane] is flipped to [membrane, nucleus]
        if img.ndim == 3 and img.shape[-1] == 2:
            img_in = img[..., ::-1]
        else:
            img_in = img
        out = model.eval(img_in, channels=cfg["classic_channels"],
                         batch_size=cfg["batch_size"], **common)
    else:
        # Cellpose-SAM 4.x: CellposeModel loads cpsam by default and needs no channels
        model = models.CellposeModel(gpu=cfg["use_gpu"])
        try:
            out = model.eval(img, batch_size=cfg["batch_size"], **common)
        except TypeError:
            # Signature differences between versions: retry without possibly unsupported kwargs
            out = model.eval(img, **common)

    masks = out[0] if isinstance(out, (list, tuple)) else out
    masks = np.squeeze(np.asarray(masks))
    return masks.astype(np.int32, copy=False)

## 5. Build the reference mask

The pixel size is read from `experiment.xenium`, and the official cell-boundary polygons (in micrometres) are rasterized into a label mask on the same pixel grid as the prediction.

In [ ]:
def read_pixel_size(experiment_path, default=XENIUM_DEFAULT_PIXEL_SIZE):
    """Recursively search experiment.xenium / experiment_roi.json (both JSON) for pixel_size (um per pixel)."""
    if not experiment_path or not os.path.exists(experiment_path):
        warnings.warn(f"{experiment_path} not found, using the default pixel_size {default}")
        return default
    try:
        with open(experiment_path, "r") as f:
            data = json.load(f)
    except Exception as e:
        warnings.warn(f"Failed to read {experiment_path} ({e}), using the default pixel_size {default}")
        return default

    found = []

    def _walk(o):
        if isinstance(o, dict):
            for k, v in o.items():
                if isinstance(k, str) and "pixel_size" in k.lower() and isinstance(v, (int, float)):
                    found.append(float(v))
                _walk(v)
        elif isinstance(o, list):
            for v in o:
                _walk(v)

    _walk(data)
    if found:
        return found[0]
    warnings.warn(f"No pixel_size field in {experiment_path}, using the default {default}")
    return default


def _boundary_columns(df):
    """Locate the (cell_id, x, y) columns of a boundary table, allowing for common naming variants."""
    cols = {c.lower(): c for c in df.columns}
    id_col = next((cols[c] for c in ("cell_id", "label_id", "label", "id") if c in cols), None)
    # Prefer local (ROI-relative) coordinates and fall back to global coordinates
    x_col = next((cols[c] for c in ("vertex_x_local_um", "vertex_x_local", "vertex_x", "x", "x_location", "vertex_x_um") if c in cols), None)
    y_col = next((cols[c] for c in ("vertex_y_local_um", "vertex_y_local", "vertex_y", "y", "y_location", "vertex_y_um") if c in cols), None)
    return id_col, x_col, y_col


def rasterize_boundaries(parquet_path, out_shape, pixel_size_eff, offset=(0, 0), verbose=True):
    """Rasterize Xenium cell_boundaries.parquet (polygons in micrometres) into an int32 label mask.

    out_shape       : (H, W) of the output mask (the crop size when a crop is used)
    pixel_size_eff  : micrometres per pixel at this level = base pixel size x downsample factor
    offset          : (row0, col0), top-left corner of the crop in full-image pixels; (0, 0) without a crop
    Pixel coordinate = micrometres / pixel_size_eff - offset; parts outside the image are clipped by skimage.
    """
    row0, col0 = offset
    df = pd.read_parquet(parquet_path)
    id_col, x_col, y_col = _boundary_columns(df)
    if id_col is None or x_col is None or y_col is None:
        raise ValueError(f"{parquet_path}: could not identify the columns (cell_id/vertex_x/vertex_y), found {list(df.columns)}")

    H, W = out_shape
    mask = np.zeros((H, W), dtype=np.int32)

    xs_all = df[x_col].to_numpy(dtype=np.float64) / pixel_size_eff - col0   # column (x), minus the crop offset
    ys_all = df[y_col].to_numpy(dtype=np.float64) / pixel_size_eff - row0   # row (y), minus the crop offset
    ids_all = df[id_col].to_numpy()

    # factorize assigns each row a cell code in order of appearance; a stable sort by code then makes
    # the vertices of each cell contiguous while preserving their original (winding) order.
    # This is O(N log N) and avoids scanning all rows once per cell.
    codes, uniq = pd.factorize(ids_all)
    n_cells = len(uniq)
    sort_idx = np.argsort(codes, kind="stable")
    codes_s = codes[sort_idx]
    xs_s = xs_all[sort_idx]
    ys_s = ys_all[sort_idx]
    seg_start = np.searchsorted(codes_s, np.arange(n_cells), side="left")
    seg_start = np.append(seg_start, len(codes_s))

    label = 0
    for c in range(n_cells):
        s, e = seg_start[c], seg_start[c + 1]
        if e - s < 3:
            continue
        label += 1
        rr, cc = sk_polygon(ys_s[s:e], xs_s[s:e], shape=(H, W))   # clipped to the image automatically
        if rr.size:
            mask[rr, cc] = label
        if verbose and label % 50000 == 0:
            print(f"    rasterized {label} cells ...")
    if verbose:
        print(f"    rasterization finished, {label} cells -> mask {mask.shape}")
    return mask

## 6. Align the shapes and evaluate

The reference mask is passed to the reference slot of `test.py` and the Cellpose prediction to the prediction slot, and the output fields are renamed so that they are unambiguous.

In [ ]:
def align_shapes(gt, pred):
    """The reference and prediction should share the same grid. If rounding makes them differ by 1-2 pixels, pad the bottom-right with zeros."""
    if gt.shape == pred.shape:
        return gt, pred
    warnings.warn(f"Reference {gt.shape} and prediction {pred.shape} differ in shape; padding the bottom-right with zeros (check that they are truly aligned)")
    H = max(gt.shape[0], pred.shape[0])
    W = max(gt.shape[1], pred.shape[1])

    def _pad(m):
        out = np.zeros((H, W), dtype=m.dtype)
        out[:m.shape[0], :m.shape[1]] = m
        return out
    return _pad(gt), _pad(pred)


def evaluate(gt_mask, pred_mask, dataset_name, gt_source,
             boundary_mode=BOUNDARY_MODE, iou_threshold=IOU_THRESHOLD):
    """Compute the metrics of test.py with the reference in the reference slot and Cellpose in the prediction slot."""
    gt_mask, pred_mask = align_shapes(gt_mask, pred_mask)

    raw = metrics.evaluate_single_mask_against_cellpose(
        method_name="Cellpose",
        cellpose_gt_mask=gt_mask,     # reference mask goes here
        pred_mask=pred_mask,          # Cellpose prediction goes here
        boundary_mode=boundary_mode,
        iou_threshold=iou_threshold,
    )

    result = {
        "dataset": dataset_name,
        "gt_source": gt_source,       # manual_labels / xenium_official_boundaries
        "method": "Cellpose",
        "gt_cell_count": raw.pop("cellpose_reference_cell_count"),   # renamed: this is the number of reference cells
        "pred_cell_count": raw.pop("pred_cell_count"),
    }
    raw.pop("reference", None)        # drop the misleading "reference": "Cellpose"
    raw.pop("method", None)
    raw.pop("mDA_note", None)
    result.update(raw)
    return result

## 7. Pipeline for one dataset

The steps are reading the morphology channels, cropping, running Cellpose, building the reference mask in the same crop window, saving both masks as int32 TIFF files, and computing the metrics.

In [ ]:
def _apply_crop(arr, crop):
    if crop is None:
        return arr
    y0, y1, x0, x1 = crop
    return arr[y0:y1, x0:x1]


def run_dataset(cfg, out_dir):
    name = cfg["name"]
    ddir = cfg["dataset_dir"]
    print(f"\n=== [{name}] {ddir} ===")

    # 1) Read the morphology channels and assemble the Cellpose input (full image)
    morph = load_morphology_channels(ddir, pyramid_level=cfg.get("pyramid_level", 0))
    factor = morph["factor"]
    print(f"  channels={len(morph['channels'])}  names={morph['names']}  downsample factor={factor:.3f}")

    img_full, in_mode = assemble_cellpose_input(morph["channels"], use_membrane=cfg.get("use_membrane", True))
    Himg, Wimg = img_full.shape[:2]

    # Crop window: an explicit crop=(y0, y1, x0, x1) has priority; otherwise crop_size selects a centred square
    crop = cfg.get("crop")
    if crop is None and cfg.get("crop_size"):
        s = int(cfg["crop_size"])
        y0 = max((Himg - s) // 2, 0)
        x0 = max((Wimg - s) // 2, 0)
        crop = (y0, min(y0 + s, Himg), x0, min(x0 + s, Wimg))
        print(f"  crop_size={s} -> centred crop={crop} (full image {Himg}x{Wimg})")

    img = _apply_crop(img_full, crop)
    del img_full
    seg_shape = img.shape[:2]
    print(f"  Cellpose input: shape={img.shape} mode={in_mode}")

    # 2) Cellpose segmentation (prediction)
    print("  Running Cellpose ...")
    pred = run_cellpose(img, CELLPOSE)
    print(f"  predicted cells={len(metrics.get_instance_ids(pred))}  mask={pred.shape}")

    # 3) Build the reference mask in the same crop window as the prediction
    if cfg["gt_type"] == "xenium_boundaries":
        exp_path = os.path.join(ddir, cfg.get("experiment", "experiment.xenium"))
        base_px = read_pixel_size(exp_path)
        px_eff = base_px * factor
        print(f"  pixel_size={base_px} um/px  effective pixel_size at this level={px_eff:.4f} um/px")
        if crop is not None:
            y0, y1, x0, x1 = crop
            # Rasterize the polygons directly into the crop coordinate system (subtract the top-left offset and
            # output only the crop size). This avoids building a full-image mask and stays correct for any window.
            gt = rasterize_boundaries(os.path.join(ddir, cfg["boundaries"]),
                                      out_shape=(y1 - y0, x1 - x0),
                                      pixel_size_eff=px_eff, offset=(y0, x0))
        else:
            gt = rasterize_boundaries(os.path.join(ddir, cfg["boundaries"]),
                                      out_shape=seg_shape, pixel_size_eff=px_eff)
        gt_source = "xenium_official_boundaries"
    else:
        raise ValueError(f"Unknown gt_type: {cfg['gt_type']}")
    print(f"  reference cells={len(metrics.get_instance_ids(gt))}  source={gt_source}")

    # 4) Save the masks (int32 TIFF, readable by read_tif_mask in test.py) for reproducibility
    os.makedirs(out_dir, exist_ok=True)
    pred_path = os.path.join(out_dir, f"{name}_cellpose_pred.tif")
    gt_path = os.path.join(out_dir, f"{name}_gt.tif")
    tifffile.imwrite(pred_path, pred)
    tifffile.imwrite(gt_path, gt)
    print(f"  saved {pred_path} / {gt_path}")

    # 5) Metrics
    return evaluate(gt, pred, dataset_name=name, gt_source=gt_source)

## 8. Run all datasets and save the summary table

In [ ]:
import traceback

os.makedirs(OUTPUT_DIR, exist_ok=True)
rows = []
for cfg in DATASETS:
    try:
        rows.append(run_dataset(cfg, OUTPUT_DIR))
    except Exception as e:
        print(f"[ERROR] dataset {cfg['name']} failed: {e}")
        traceback.print_exc()

if not rows:
    print("No dataset finished successfully.")
else:
    df = pd.DataFrame(rows)
    csv_path = os.path.join(OUTPUT_DIR, "cellpose_xenium_metrics.csv")
    df.to_csv(csv_path, index=False)

    pd.set_option("display.max_columns", None, "display.width", 200)
    print("\n================ Summary ================")
    print(df.to_string(index=False))
    print("\nSaved:", csv_path)
    display(df)